In [0]:
import os
import json
import re
import zipfile
from databricks.sdk import WorkspaceClient

# 1. Initialize the client
w = WorkspaceClient()

print("Starting Databricks Dashboard export...")

# Set up the target folder path
workspace_folder_path = "/Workspace/Users/ibhalla@infomedia.com.au/Notebooks/Dashboard/"
os.makedirs(workspace_folder_path, exist_ok=True)

# 2. Loop through all AI/BI Dashboards (Lakeview API)
for dash_summary in w.lakeview.list():
    try:
        # Fetch the full dashboard details
        dash = w.lakeview.get(dashboard_id=dash_summary.dashboard_id)
        
        # Clean up the name for the file
        safe_name = re.sub(r'[^\w\-_\.]', '_', dash.display_name)
        
        # Define the zip file path and the name of the file inside the zip
        zip_filepath = os.path.join(workspace_folder_path, f"{safe_name}.zip")
        internal_filename = f"{safe_name}.lvdash.json"
        
        if dash.serialized_dashboard:
            # Parse and format the JSON so it looks nice if someone opens it manually
            dash_data = json.loads(dash.serialized_dashboard)
            pretty_json = json.dumps(dash_data, indent=2)
            
            # Create a zip file and write the JSON string directly into it
            with zipfile.ZipFile(zip_filepath, 'w', zipfile.ZIP_DEFLATED) as zf:
                zf.writestr(internal_filename, pretty_json)
                
            print(f"✅ Exported & Zipped: {dash.display_name} -> {zip_filepath}")
        else:
            print(f"⚠️ Skipped {dash.display_name}: No configuration found.")
            
    except Exception as e:
        print(f"❌ Error exporting {dash_summary.display_name}: {e}")